In [45]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [46]:
data_folder = "../../data/"

# Create table with groups and column names per keyword

In [47]:
# Load table with the original keywords downloaded from old github repo
original_keyword_df = pd.read_csv(data_folder + "keyword_list.csv")

In [48]:
original_keyword_df.rename(columns={"label":"keyword"}, inplace=True)

In [49]:
# Load with table of translated keywords
translation_keyword_df = pd.read_excel(data_folder + "FS Predictions - Keywords - ENG - ARB.xlsx")

In [50]:
# Join the two tables
keyword_df = translation_keyword_df.merge(original_keyword_df, left_on="English", right_on="keyword", how="left")

In [51]:
# Fill NAs in the the OLD keyword part of the joint table from above
keyword_df.iloc[:209] = keyword_df.iloc[:209].ffill()
keyword_df = keyword_df.drop(columns=["ID ", "Comment", "keyword"])

# Add the new keyword category "New Category" to the new keyword part of the joint table
keyword_df.loc[keyword_df["cluster"].isna(),"cluster"] = "New Category"

# Fill NAs in the NEW keyword part from above
keyword_df = keyword_df.ffill()

In [52]:
# Make sure that there are no leading or trailing whitespaces
keyword_df.loc[~keyword_df["English"].isna(), "English"] = keyword_df.loc[~keyword_df["English"].isna(), "English"].apply(lambda x: x.strip())
keyword_df.loc[~keyword_df["Arabic"].isna(), "Arabic"] = keyword_df.loc[~keyword_df["Arabic"].isna(), "Arabic"].apply(lambda x: x.strip())

In [53]:
keyword_categories = {'CONFLICTS AND VIOLENCE': 'cv',
                    'POLITICAL INSTABILITY': 'pi',
                    'HUMANITARIAN AID': 'ha',
                    'ECONOMIC ISSUES': 'eci',
                    'AGRICULTURAL PRODUCTION ISSUES': 'prs',
                    'WEATHER SHOCKS': 'ws',
                    'FOOD CRISIS': 'fc',
                    'LAND-RELATED ISSUES': 'lri',
                    'PESTS AND DISEASES': 'pad',
                    'FORCED DISPLACEMENT': 'fd',
                    'ENVIRONMENTAL ISSUES': 'ei',
                    'OTHER': 'o',
                    'NEW CATEGORY': 'nc',
                    'ANY KEYWORD': 'kw'}

In [54]:
# Make the group names upper case
keyword_df["cluster"] = keyword_df["cluster"].str.upper()

In [55]:
# Make sure all keywords are lower case
keyword_df["English"] = keyword_df["English"].str.lower()

In [56]:
# Create "ColumnName" column
keyword_df["ColumnName1"] = keyword_df["cluster"].apply(lambda x: keyword_categories[x] + "_")
keyword_df["ColumnName2"] = keyword_df["English"].apply(lambda x: "_".join(x.split(" ")))
keyword_df["ColumnName"] = keyword_df["ColumnName1"] + keyword_df["ColumnName2"]
keyword_df = keyword_df.drop(columns=["ColumnName1", "ColumnName2"])

In [57]:
# Repeat all keywords for the Any Keyword category
keyword_df_orig = keyword_df.copy()
keyword_df["cluster"] = "ANY KEYWORD"
keyword_df = pd.concat([keyword_df_orig, keyword_df]).reset_index(drop=True)

In [58]:
# Create entries for the category summaries
summary_columns_df = pd.DataFrame(data={"English": np.nan, "Arabic": np.nan, "ColumnName" : keyword_categories.values(), "cluster": keyword_categories.keys()})

In [59]:
keyword_df = pd.concat([keyword_df, summary_columns_df]).reset_index(drop=True)

In [60]:
keyword_df.rename(columns={"English": "keyword_eng", "Arabic": "keyword_ara", "cluster": "group_name", "ColumnName": "column_name"}, inplace=True)
keyword_df.reset_index(drop=True, inplace=True)

In [61]:
keyword_df.to_csv(data_folder + "keywords_dataframe.csv", index=False)